# v0.16.0 — Connection-level authentication

Authenticate the ORM's connection as a SurrealDB **record user** or as a different **system
user**, keep that identity across reconnects, and read back who is signed in.

This notebook walks every public entry point the version adds:

| Method | What it does |
| ------ | ------------ |
| `signup(access=, variables=)` | register a new record user |
| `signin(access=, variables=)` | authenticate an existing record user |
| `signin(username=, password=)` | authenticate a system user |
| `signin(access=, refresh=)` | renew a session without the password — **SurrealDB 3.x only** |
| `authenticate(token)` | re-attach a JWT on a later request |
| `info(return_type=)` | who am I? |
| `invalidate()` | log out, back to the application's own identity |

**2.6.x vs 3.x**: the five methods behave identically on both lines. Only refresh tokens differ
— `DEFINE ACCESS … WITH REFRESH` does not parse on 2.6.x, so `tokens.refresh` is always `None`
there. Section 8 probes the server and explains instead of failing.

## 1. Connect

The URL is env-overridable so you can point this at your own server.

In [1]:
import contextlib
import os
from uuid import uuid4

from typing import Any

from surreal_orm_lite import (
    BaseSurrealModel,
    SurrealConfigDict,
    SurrealDBConnectionManager,
    SurrealDbAuthenticationError,
)

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")
SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root", password="root",
    namespace="examples", database="examples",
)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


## 2. Declare the access method

Authentication needs a `DEFINE ACCESS … TYPE RECORD` method. Declaring it is DDL, so it goes
through `query()` — a `define_access()` helper is deliberately left to the `schema.py` module
the roadmap opens at v0.31.0.

Note the **table permissions**. Without `FOR select`, a record user cannot read *itself*, and
`info()` returns `None` even though the signin succeeded. That is the quietest trap of this
feature, and section 6 demonstrates it on purpose.

In [2]:
# Always fetch the client where you use it, never cache it in a notebook variable:
# reconnect() (section 8) replaces it, and a stale handle fails in confusing ways.
client = await SurrealDBConnectionManager.get_client()

await client.query(
    "DEFINE TABLE OVERWRITE AppUser SCHEMALESS "
    "PERMISSIONS FOR select, update WHERE id = $auth.id;", {},
)
await client.query("""
DEFINE ACCESS OVERWRITE account ON DATABASE TYPE RECORD
  SIGNUP ( CREATE AppUser SET email = $email, pass = crypto::argon2::generate($pass) )
  SIGNIN ( SELECT * FROM AppUser
           WHERE email = $email AND crypto::argon2::compare(pass, $pass) )
  DURATION FOR TOKEN 15m, FOR SESSION 12h;
""", {})
print("Access method 'account' defined.")

Access method 'account' defined.


## 3. A model for the record table

`info(return_type=…)` can hydrate the session record straight into a model.

In [3]:
class AppUser(BaseSurrealModel):
    # The table name is the class name, so this model maps to the `AppUser` table declared
    # above. `id` comes back as a native `RecordID`, hence `Any` rather than `str`.
    model_config = SurrealConfigDict(primary_key="id")

    id: Any = None
    email: str


# A fresh identifier per run keeps this notebook re-runnable. It also avoids a real pitfall:
# with two records sharing the signin identifier, SurrealDB 2.6.x fails the signin outright
# while 3.x picks one.
EMAIL = f"{uuid4().hex[:12]}@example.test"
PASSWORD = "s3cret-passphrase"
print("Demo user:", EMAIL)

Demo user: 5bc5ea77526a@example.test


## 4. `signup()` — register a record user

The connection is left authenticated as the newly created record.

In [4]:
tokens = await SurrealDBConnectionManager.signup(
    access="account", variables={"email": EMAIL, "pass": PASSWORD},
)

print("type:", type(tokens).__name__)
print("has an access token:", bool(tokens.access))
print("refresh token:", tokens.refresh)
print("repr:", repr(tokens))

type: AuthTokens
has an access token: True
refresh token: None
repr: AuthTokens(access=<redacted>, refresh=None)


Note the `repr`: **both tokens are redacted**, while it still tells you whether a refresh token
is present. There is deliberately no `__str__` returning the JWT either — a token that lands in
a log line, a traceback or a test assertion diff is a leaked credential. Read `tokens.access`
explicitly when you mean to.

## 5. `signin()` — authenticate an existing user

`namespace` and `database` come from `set_connection()`, because record access requires them.

In [5]:
# Drop back to root first, so the signin below is what actually proves the identity.
await SurrealDBConnectionManager.signin(username="root", password="root")

tokens = await SurrealDBConnectionManager.signin(
    access="account", variables={"email": EMAIL, "pass": PASSWORD},
)
print("signed in:", bool(tokens.access))
print("$auth is now:", await (await SurrealDBConnectionManager.get_client()).query("RETURN $auth.email;", {}))

signed in: True
$auth is now: 5bc5ea77526a@example.test


A **system user** is different: it gets no namespace/database by default. A root user is defined
at neither level, so sending them would turn a root signin into a database-user signin against a
user the server does not have. Pass `namespace=` (and `database=`) explicitly for a
namespace- or database-level user.

```python
await SurrealDBConnectionManager.signin(username="root", password="root")               # root
await SurrealDBConnectionManager.signin(username="u", password="p", namespace="examples")  # NS
```

## 6. `info()` — who am I?

Raw, or hydrated into a model.

In [6]:
raw = await SurrealDBConnectionManager.info()
print("raw keys:", sorted(raw))

me = await SurrealDBConnectionManager.info(return_type=AppUser)
print("as a model:", type(me).__name__, "->", me.email)

raw keys: ['email', 'id', 'pass']
as a model: AppUser -> 5bc5ea77526a@example.test


In [7]:
# Careful — this does NOT report None, and that surprises people.
await SurrealDBConnectionManager.signin(username="root", password="root")
still_reported = await SurrealDBConnectionManager.info()
print("info() after switching to root:", None if still_reported is None else still_reported["email"])
print("permissions really are root's:", bool(await (await SurrealDBConnectionManager.get_client()).query("INFO FOR DB;", {})))

info() after switching to root: 5bc5ea77526a@example.test
permissions really are root's: True


Signing in as a system user swaps the **permissions** but leaves `$auth` pointing at the record,
so `info()` keeps reporting it. Measured identically on SurrealDB 2.6.5 and 3.2.4. Only
`invalidate()` (section 7) really ends a record session — a root re-signin is not a logout.

> ⚠️ `info()` also returns `None` when the record's table does **not** grant the record user
> `select` on itself. The signin succeeded and `$auth` is set, but the server hands back
> nothing — no error. If `info()` surprises you with a `None`, check the table permissions
> before suspecting your credentials.
>
> Because the cause is a permission rather than a bad payload, a `None` is passed through
> untouched even when you pass `return_type=` — failing validation there would blame the model
> for a permissions problem.

## 7. `authenticate()` and `invalidate()`

The web round trip: hand `tokens.access` to a browser, get it back on the next request.

In [8]:
stored_jwt = tokens.access  # what you would put in a web session

await SurrealDBConnectionManager.invalidate()
print("after invalidate(), info():", await SurrealDBConnectionManager.info())
print("the connection still works:", bool(await (await SurrealDBConnectionManager.get_client()).query("INFO FOR DB;", {})))

await SurrealDBConnectionManager.authenticate(stored_jwt)
print("after authenticate(), info():", (await SurrealDBConnectionManager.info())["email"])

after invalidate(), info(): None
the connection still works: True
after authenticate(), info(): 5bc5ea77526a@example.test


Two things worth knowing:

- **`invalidate()` returns the connection to the identity you configured**, rather than leaving
  it anonymous. The SDK's own `invalidate()` leaves the session anonymous — and since the ORM
  caches one client per event loop, every later call on that connection would fail. Hence the
  `INFO FOR DB` above still working.
- **`invalidate()` is the only real logout.** Signing in as a system user swaps the permissions
  but leaves `$auth` pointing at the record, so `info()` would keep reporting it.

## 8. Session persistence across reconnects

A client belongs to the event loop that created it, and is dropped when that loop ends. Without
replaying the token, a reconnect would silently hand you back the configured **root** identity —
a privilege change nobody asked for. The ORM replays it instead.

In [9]:
before = (await SurrealDBConnectionManager.info())["email"]
await SurrealDBConnectionManager.reconnect()
after = (await SurrealDBConnectionManager.info())["email"]

print("identity before reconnect:", before)
print("identity after  reconnect:", after)
print("survived:", before == after)

identity before reconnect: 5bc5ea77526a@example.test
identity after  reconnect: 5bc5ea77526a@example.test
survived: True


If the stored token has been revoked or has expired, the replay raises
`SurrealDbAuthenticationError` — **not** a connection error, because the connection is fine and
only the identity is gone. The dead token is forgotten, so the next call connects normally at
the configured identity instead of failing forever.

## 9. Refresh tokens — SurrealDB 3.x only

`DEFINE ACCESS … WITH REFRESH` lets you renew an access token without asking for the password
again. SurrealDB 2.6.x cannot parse the clause at all, so this section probes the server first
and explains rather than failing.

In [10]:
await SurrealDBConnectionManager.signin(username="root", password="root")


async def refresh_supported() -> bool:
    """True if the server understands DEFINE ACCESS … WITH REFRESH (SurrealDB 3.x).

    Only a *parse* failure means "unsupported". Anything else is re-raised: a probe that
    reports every error as "not supported" would quietly lie to a reader on 3.x.
    """
    live = await SurrealDBConnectionManager.get_client()
    try:
        await live.query("""
        DEFINE ACCESS OVERWRITE account_refresh ON DATABASE TYPE RECORD
          SIGNUP ( CREATE AppUser SET email = $email, pass = crypto::argon2::generate($pass) )
          SIGNIN ( SELECT * FROM AppUser
                   WHERE email = $email AND crypto::argon2::compare(pass, $pass) )
          WITH REFRESH DURATION FOR TOKEN 15m, FOR SESSION 12h, FOR GRANT 30d;
        """, {})
        return True
    except Exception as exc:
        if "bearer" in str(exc).lower() or "parse error" in str(exc).lower():
            return False
        raise


supported = await refresh_supported()
print("Refresh tokens available (SurrealDB 3.x):", supported)

Refresh tokens available (SurrealDB 3.x): True


In [11]:
if supported:
    first = await SurrealDBConnectionManager.signup(
        access="account_refresh",
        variables={"email": f"{uuid4().hex[:12]}@example.test", "pass": PASSWORD},
    )
    print("signup issued a refresh token:", bool(first.refresh))

    renewed = await SurrealDBConnectionManager.signin(
        access="account_refresh", refresh=first.refresh,
    )
    print("renewed without the password:", bool(renewed.access))
    print("still the same record:", (await SurrealDBConnectionManager.info())["email"])
    print("the refresh token ROTATED:", renewed.refresh != first.refresh)

    # The spent token is dead the moment the exchange succeeds.
    try:
        await SurrealDBConnectionManager.signin(
            access="account_refresh", refresh=first.refresh,
        )
        print("!! the spent refresh token was accepted (unexpected)")
    except SurrealDbAuthenticationError:
        print("the SPENT refresh token is rejected:", True)
else:
    print(
        "On SurrealDB 2.6.x, DEFINE ACCESS … WITH REFRESH does not parse "
        "(\"expected the experimental bearer access feature to be enabled\"), so "
        "tokens.refresh is always None and signin(refresh=…) has nothing to exchange.\n"
        "Renew by signing in again with the credentials, or widen DURATION FOR TOKEN."
    )

signup issued a refresh token: True
renewed without the password: True


still the same record: 800345e6efd3@example.test
the refresh token ROTATED: True
the SPENT refresh token is rejected: True


> ⚠️ **Refresh tokens rotate.** A successful exchange kills the token it spent, immediately and
> permanently. Persist the new `tokens.refresh` before the next request — if you keep the old
> value and drop the new one, the user is logged out for good, and nothing is raised at the
> moment you make the mistake.

## 10. Errors — one exception, both DB lines

The SDK reports authentication failures inconsistently: the same wrong password raises
`NotFoundError` on 3.x and `InternalError` on 2.6.x, and a malformed token is rejected
client-side as a plain `ValueError` that never reaches a server. The ORM normalises all of them
to `SurrealDbAuthenticationError`, which subclasses `SurrealDbError` — so match on the type,
never on the server's wording.

In [12]:
cases = {
    "wrong password": dict(access="account", variables={"email": EMAIL, "pass": "wrong"}),
    "unknown access method": dict(access="nope", variables={"email": EMAIL, "pass": PASSWORD}),
}
for label, kwargs in cases.items():
    try:
        await SurrealDBConnectionManager.signin(**kwargs)
        print(f"{label}: no error (unexpected)")
    except SurrealDbAuthenticationError as exc:
        print(f"{label}: {type(exc).__name__}")

try:
    await SurrealDBConnectionManager.authenticate("not-a-jwt")
except SurrealDbAuthenticationError as exc:
    print("malformed token:", type(exc).__name__)

wrong password: SurrealDbAuthenticationError
unknown access method: SurrealDbAuthenticationError
malformed token: SurrealDbAuthenticationError


In [13]:
# Invalid argument combinations are caught before any request is issued.
for label, kwargs in {
    "no credentials at all": {},
    "system user mixed with record access": dict(username="root", password="root", access="account"),
    "variables and refresh together": dict(access="account", variables={"a": 1}, refresh="x"),
}.items():
    try:
        await SurrealDBConnectionManager.signin(**kwargs)
    except ValueError as exc:
        print(f"{label}: ValueError — {str(exc).split('.')[0]}.")

no credentials at all: ValueError — No credentials given.
system user mixed with record access: ValueError — Pass either system-user credentials (username=, password=) or record access (access= with variables= or refresh=), not both — they are two different identities.
variables and refresh together: ValueError — Pass either variables= or refresh=, not both — variables= proves an identity from scratch, refresh= renews one that already exists.


## 11. Cleanup

In [14]:
await SurrealDBConnectionManager.signin(username="root", password="root")
client = await SurrealDBConnectionManager.get_client()
for statement in (
    "REMOVE ACCESS account ON DATABASE;",
    "REMOVE ACCESS account_refresh ON DATABASE;",
    "REMOVE TABLE AppUser;",
):
    with contextlib.suppress(Exception):
        await client.query(statement, {})

await SurrealDBConnectionManager.unset_connection()
print("Cleaned up. Session token cleared:", SurrealDBConnectionManager.get_session_token() is None)

Cleaned up. Session token cleared: True


## Takeaways

- One of **three** credential shapes per `signin()`: record access, system user, or refresh
  exchange. Namespace/database default to the configured connection **for record access only**.
- `AuthTokens` redacts on `repr` and offers no JWT-returning `__str__` — read `.access` on purpose.
- The identity **survives reconnects**; a revoked token raises an *authentication* error, not a
  connection error, and is dropped so the next call still works.
- `invalidate()` is the only real logout, and it leaves the connection usable.
- `info()` returning `None` usually means a table permission, not bad credentials.
- Refresh tokens are 3.x-only and **rotate** — persist the new one.
- Auth changes the identity of the **whole connection**; every model shares it.